In [ ]:
import pandas as pd
df = pd.read_csv("noticias_terca.csv", sep=",", encoding="utf-8")
print('\nMostrando os 5 primeiros registros:')
pd.options.display.max_columns = None
print(df.head(5))

In [ ]:
print('\nMostrando as informações do DataFrame:')
df.info()
print('\nMostrando Labels:')
print(df['fellingtarget'].value_counts())

In [ ]:
print(df.query("typetarget == 'poluição' and fellingtarget == 1").head())

In [ ]:
# Mostrando correlação entre variáveis
df['len_text'] = df['text'].str.len()
df['len_title'] = df['title'].str.len()
df['num_palavras'] = df['text'].str.split().apply(len)
import seaborn as sns
import numpy as np
df_correlacao = df.corr(numeric_only=True)
sns.heatmap(df_correlacao, cmap='RdBu'
, fmt='.2f', square=True, annot=True)

In [ ]:
# Separando os dados de teste (30%) e de treino (70%)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Retirando as 4 colunas menos importantes
x_data = df.drop(["typetarget"
,
"fellingtarget"
,
"title"
,
"text"
,
"url"], axis=1, inplace=False)
y_data = df["typetarget"]
x_train, x_test, y_train, y_test = train_test_split(
x_data, y_data, test_size=0.3, random_state=42)
# NORMALIZAÇÃO: Crucial para algoritmos sensíveis à escala
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
print("x train: "
, x_train.shape)
print("x test: "
, x_test.shape)
print("y train: "
, y_train.shape)
print("y test: "
, y_test.shape)
print("✓ Dados normalizados com sucesso!")

In [ ]:
# Verificar e remover valores NaN
print("Verificando valores NaN nos dados...")
print(f"NaNs em x_train: {np.isnan(x_train).sum()}")
print(f"NaNs em x_test: {np.isnan(x_test).sum()}")

# Remover linhas com NaN
mask_train = ~np.isnan(x_train).any(axis=1)
mask_test = ~np.isnan(x_test).any(axis=1)

x_train = x_train[mask_train]
x_test = x_test[mask_test]
y_train = y_train[mask_train]
y_test = y_test[mask_test]

print(f"\nApos remover NaNs:")
print(f"x train: {x_train.shape}")
print(f"x test: {x_test.shape}")
print(f"y train: {y_train.shape}")
print(f"y test: {y_test.shape}")
print("✓ NaNs removidos com sucesso!")

In [ ]:
# Relatório de Desempenho com Validação Cruzada
from sklearn.metrics import classification_report, confusion_matrix
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from time import time
def mostrar_desempenho(x_train, y_train, x_test, y_test, model, name):
    # Treinando modelo
    inicio = time()
    model.fit(x_train, y_train)
    fim = time()
    tempo_treinamento = (fim - inicio)*1000
    # Prevendo dados
    inicio = time()
    y_predicted = model.predict(x_test)
    fim = time()
    tempo_previsao = (fim - inicio)*1000
    # Validação Cruzada (5-fold)
    cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring='accuracy')
    print('='*60)
    print('Relatório Utilizando Algoritmo:', name)
    print('='*60)
    print('\nMostrando Matriz de Confusão:')
    print(confusion_matrix(y_test, y_predicted))
    print('\nMostrando Relatório de Classificação:')
    print(metrics.classification_report(y_test, y_predicted))
    accuracy = accuracy_score(y_test, y_predicted)
    print('Accuracy (teste):', accuracy)
    print('Accuracy (validação cruzada 5-fold):', f"{cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    relatorio = metrics.classification_report(y_test, y_predicted, output_dict=True)
    print('Precision:', relatorio['macro avg']['precision'])
    print('Tempo de treinamento (ms):', tempo_treinamento)
    print('Tempo de previsão (ms):', tempo_previsao)
    return accuracy, tempo_treinamento, tempo_previsao

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
model_mlp = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=10000)
acc_mlp, tt_mlp, tp_mlp = mostrar_desempenho(
x_train, y_train, x_test, y_test, model_mlp, 'MLP')

In [ ]:
# Classificação com DecisionTree
from sklearn import tree
model_arvore = tree.DecisionTreeClassifier()
acc_dt, tt_dt, tp_dt = mostrar_desempenho(
x_train, y_train, x_test, y_test, model_arvore,
'DecisionTree')

In [ ]:
# Classificação com RandomForest
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(max_depth=10, n_estimators=100, max_features='sqrt', random_state=42)
acc_rf, tt_rf, tp_rf = mostrar_desempenho(
x_train, y_train, x_test, y_test, model_rf, 'RandomForest')

In [ ]:
# Classificação com AdaBoost
from sklearn.ensemble import AdaBoostClassifier
acc_ada, tt_ada, tp_ada = mostrar_desempenho(
x_train, y_train, x_test, y_test, AdaBoostClassifier(), 'AdaBoost')

In [ ]:
# Classificação com KNN
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier(n_neighbors=5, p=2)
acc_knn, tt_knn, tp_knn = mostrar_desempenho(
x_train, y_train, x_test, y_test, model_knn, 'KNN')

In [ ]:
# Classificação com Regressão Logística
from sklearn.linear_model import LogisticRegression
acc_lr, tt_lr, tp_lr = mostrar_desempenho(
x_train, y_train, x_test, y_test, LogisticRegression(), 'LR')

In [ ]:
# Classificação com SVM
from sklearn.svm import SVC
acc_svm, tt_svm, tp_svm = mostrar_desempenho(
x_train, y_train, x_test, y_test, SVC(), 'SVM')

In [ ]:
# Classificação com GaussianNB
from sklearn.naive_bayes import GaussianNB
acc_gnb, tt_gnb, tp_gnb = mostrar_desempenho(
x_train, y_train, x_test, y_test, GaussianNB(), 'GaussianNB')

In [ ]:
# Classificação com LDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
acc_lda, tt_lda, tp_lda = mostrar_desempenho(
x_train, y_train, x_test, y_test, LinearDiscriminantAnalysis(), 'LDA')

In [ ]:
# Classificação com QDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
model_qda = QuadraticDiscriminantAnalysis()
acc_qda, tt_qda, tp_qda = mostrar_desempenho(
x_train, y_train, x_test, y_test, model_qda, 'QDA')

In [ ]:
# Classificação com SGDClassifier (Modelo da IA em Produção)
# Nota: A IA usa TfidfVectorizer + SGDClassifier em um Pipeline
# Aqui testamos em features numéricas já normalizadas
from sklearn.linear_model import SGDClassifier
model_sgd = SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42)
acc_sgd, tt_sgd, tp_sgd = mostrar_desempenho(
x_train, y_train, x_test, y_test, model_sgd, 'SGDClassifier (IA em Produção)')


In [ ]:
# Comparação de Desempenho em Accuracy
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(14, 6))
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])
algoritmos = ['GaussianNB', 'MLP', 'DecisionTree', 'KNN', 'LogisticRegression', 'LDA', 'SVM', 'RandomForest', 'AdaBoost', 'QDA', 'SGDClassifier (IA)']
accs = [acc_gnb, acc_mlp, acc_dt, acc_knn, acc_lr, acc_lda, acc_svm, acc_rf, acc_ada, acc_qda, acc_sgd]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#ff0000']
bars = ax.bar(algoritmos, accs, color=colors, edgecolor='black', linewidth=1.2)
ax.set_xlabel('Algoritmo', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Comparação de Desempenho - Accuracy dos Modelos', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
plt.xticks(rotation=45, ha='right')
# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Comparação de Desempenho em Tempo de Treinamento
fig = plt.figure(figsize=(14, 6))
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])
tts = [tt_gnb, tt_mlp, tt_dt, tt_knn, tt_lr, tt_lda, tt_svm, tt_rf, tt_ada, tt_qda, tt_sgd]
bars = ax.bar(algoritmos, tts, color=colors, edgecolor='black', linewidth=1.2)
ax.set_xlabel('Algoritmo', fontsize=12, fontweight='bold')
ax.set_ylabel('Tempo de Treinamento (ms)', fontsize=12, fontweight='bold')
ax.set_title('Comparação de Desempenho - Tempo de Treinamento', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Comparação de Desempenho em Tempo de Previsão
fig = plt.figure(figsize=(14, 6))
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])
tps = [tp_gnb, tp_mlp, tp_dt, tp_knn, tp_lr, tp_lda, tp_svm, tp_rf, tp_ada, tp_qda, tp_sgd]
bars = ax.bar(algoritmos, tps, color=colors, edgecolor='black', linewidth=1.2)
ax.set_xlabel('Algoritmo', fontsize=12, fontweight='bold')
ax.set_ylabel('Tempo de Previsão (ms)', fontsize=12, fontweight='bold')
ax.set_title('Comparação de Desempenho - Tempo de Previsão', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
